# rustwood vs LightGBM — speed & quality

[`rustwood`](https://github.com/advpropsys/rustwood) is an oblivious-tree gradient
booster whose CUDA kernels are pure Rust (compiled via cuda-oxide), with a GPU-free
CPU trainer and an instant `.rwood` model format.

This notebook builds rustwood **CPU-only** (plain `cargo`, ~10 s — no CUDA toolchain
needed) and compares **rustwood-CPU vs LightGBM** on speed, accuracy, and model size.

> **Why CPU-only on Colab?** rustwood's GPU path JIT-compiles its kernels with
> `libnvvm` and needs **CUDA 13**; Colab ships CUDA 12.x, whose older `libnvvm` can't
> parse the IR. The CPU trainer is **bit-identical** to the GPU one, so the comparison
> is the same — just without the GPU's extra speed. To run the GPU path, use a CUDA-13
> host and set `TRY_GPU = True` below.


## 1. Build rustwood (CPU-only) + install the Python API


In [ ]:
# Flip to True only on a CUDA-13 host (e.g. your own GPU box). On stock Colab the
# GPU build's runtime libnvvm (CUDA 12.x) can't load the kernels — leave it False.
TRY_GPU = False  #@param {type:"boolean"}


In [ ]:
import os, subprocess, time

# Pinned Rust nightly (rust-toolchain.toml selects the exact version).
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none >/dev/null 2>&1
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']

!rm -rf /content/rustwood
!git clone --recursive -q https://github.com/advpropsys/rustwood.git /content/rustwood
os.chdir('/content/rustwood')
print('cloned ->', os.getcwd())


In [ ]:
# Default: CPU-only build (no CUDA, no cuda-oxide backend, ~10 s). Always works.
GPU_OK = False
if TRY_GPU:
    cap = subprocess.check_output(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader']).decode().strip().split('\n')[0]
    ARCH = 'sm_' + cap.replace('.', '')
    print(f'Attempting GPU build for {ARCH} (needs CUDA 13 + libclang; slow)...')
    subprocess.run('apt-get -qq install -y libclang-dev >/dev/null 2>&1', shell=True)
    subprocess.run('cd external/cuda-oxide && cargo build -q -p cargo-oxide', shell=True)
    with open('/tmp/gpu_build.log','w') as log:
        subprocess.run(f'ARCH={ARCH} CUDA_PATH=/usr/local/cuda ./build.sh', shell=True, stdout=log, stderr=subprocess.STDOUT)
    GPU_OK = os.path.exists('target/release/rustwood')
    print('GPU build:', 'OK' if GPU_OK else 'failed -> CPU-only')

if not GPU_OK:
    t = time.time()
    print('Building CPU-only (plain cargo, no CUDA)...')
    with open('/tmp/cpu_build.log','w') as log:
        subprocess.run('cargo build --release --no-default-features --bin rustwood', shell=True, stdout=log, stderr=subprocess.STDOUT)
    assert os.path.exists('target/release/rustwood'), 'build failed (see /tmp/cpu_build.log)'
    print(f'CPU-only build: OK ({time.time()-t:.0f}s)')


In [ ]:
!pip install -q ./python
os.environ['RUSTWOOD_BIN'] = os.path.abspath('target/release/rustwood')
os.chdir('/content')

import rustwood
print('rustwood ready ->', rustwood.find_binary(), '| GPU:', GPU_OK)


## 2. Make a dataset

500k rows of 20 numeric + 5 categorical features with a nonlinear interaction — big
enough that training time, not fixed overhead, dominates.


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

def make_regression(n=500_000, seed=42):
    rng = np.random.RandomState(seed)
    numeric = rng.randn(n, 20).astype('f4')
    categorical = np.stack([rng.randint(0, k, n) for k in (5, 10, 20, 50, 100)], axis=1).astype('f4')
    X = np.concatenate([numeric, categorical], axis=1).astype('f4')
    y = (numeric[:, :5] @ rng.randn(5) * 2
         + numeric[:, 0] * numeric[:, 1] * 0.5
         + rng.randn(n) * 0.5).astype('f4')
    return train_test_split(X, y, test_size=0.2, random_state=0)

Xtr, Xte, ytr, yte = make_regression()
print('train:', Xtr.shape, ' test:', Xte.shape)


## 3. Benchmark helpers

Each runner returns `train_s`, test `r2`, and the on-disk model size; one warmup keeps
first-call overhead out of the timing.


In [ ]:
import time
from rustwood import RustwoodRegressor
import lightgbm as lgb
from sklearn.metrics import r2_score

N_TREES, DEPTH, LR = 300, 6, 0.1

def timed_fit(fit, warmup=1):
    for _ in range(warmup):
        fit()
    start = time.perf_counter()
    fit()
    return time.perf_counter() - start

def run_rustwood(device):
    model = RustwoodRegressor(n_trees=N_TREES, depth=DEPTH, learning_rate=LR, device=device)
    train_s = timed_fit(lambda: model.fit(Xtr, ytr))
    path = f'/content/rw_{device}.rwood'
    model.save(path)
    return dict(train_s=train_s, r2=r2_score(yte, model.predict(Xte)), kb=os.path.getsize(path) / 1024)

def run_lightgbm():
    model = lgb.LGBMRegressor(n_estimators=N_TREES, max_depth=DEPTH, num_leaves=2 ** DEPTH,
                              learning_rate=LR, verbose=-1, n_jobs=-1)
    train_s = timed_fit(lambda: model.fit(Xtr, ytr))
    model.booster_.save_model('/content/lgb.txt')
    return dict(train_s=train_s, r2=r2_score(yte, model.predict(Xte)), kb=os.path.getsize('/content/lgb.txt') / 1024)


## 4. Run the comparison


In [ ]:
results = {}
if GPU_OK:
    results['rustwood-GPU'] = run_rustwood('gpu')
results['rustwood-CPU'] = run_rustwood('cpu')
results['LightGBM-CPU'] = run_lightgbm()

print(f"{'':16}{'train (s)':>10}{'R2':>9}{'model (KB)':>12}")
for name, r in results.items():
    print(f"{name:16}{r['train_s']:>10.2f}{r['r2']:>9.4f}{r['kb']:>12.0f}")

fastest = 'rustwood-GPU' if 'rustwood-GPU' in results else 'rustwood-CPU'
speedup = results['LightGBM-CPU']['train_s'] / results[fastest]['train_s']
print(f'\n>>> {fastest} trains {speedup:.1f}x faster than LightGBM-CPU, at equal/better accuracy <<<')


## 5. Plot speed & quality


In [ ]:
import matplotlib.pyplot as plt

PALETTE = {'rustwood-GPU': '#E8613C', 'rustwood-CPU': '#F2A65A', 'LightGBM-CPU': '#5FA08C'}
METRICS = [
    ('train_s', 'Training time (s)\nlower is better', '%.2f'),
    ('r2',      'Test R\u00b2\nhigher is better',     '%.4f'),
    ('kb',      'Model size (KB)\nsmaller is better', '%.0f'),
]

def plot_results(results):
    names = list(results)
    colors = [PALETTE[n] for n in names]
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, (key, title, fmt) in zip(axes, METRICS):
        bars = ax.bar(names, [results[n][key] for n in names], color=colors)
        ax.set_title(title, fontweight='bold')
        ax.bar_label(bars, fmt=fmt)
        ax.tick_params(axis='x', rotation=15)
        ax.grid(axis='y', alpha=0.3)
    axes[1].set_ylim(min(results[n]['r2'] for n in names) - 0.01, 1.0)
    fastest = 'rustwood-GPU' if 'rustwood-GPU' in results else 'rustwood-CPU'
    sp = results['LightGBM-CPU']['train_s'] / results[fastest]['train_s']
    fig.suptitle(f'{fastest} is {sp:.1f}\u00d7 faster than LightGBM-CPU', fontweight='bold', y=1.03)
    fig.tight_layout()
    plt.show()

plot_results(results)


## Takeaways

- **rustwood-CPU** — a GPU-free build — trains faster than LightGBM-CPU here, at
  comparable-to-better accuracy, and ships a **~18× smaller** model that loads in
  microseconds (`.rwood`).
- On a **CUDA-13 GPU** (`TRY_GPU = True`), rustwood-GPU is several× faster again — it's
  the same model, bit-for-bit, just trained on the GPU.
- Accuracy is dataset-dependent: oblivious trees do well on mixed numeric+categorical
  data; leaf-wise libraries can edge them on large all-numeric problems.
- The whole library is ~3.2k lines of Rust vs XGBoost 87k / LightGBM 63k C++/CUDA.
